# Supervised Classifier Performance Evaluation

In this notebook we will demonstrate how to evaluate the performance of a supervised classifier. This assumes that we have ground truth labels for each of our data points, and we can properly test "how good" a classifier is.

# Imports

These are standard imports for working with the data and loading our Google Drive (which contains our CSV file) -- if your data is stored on your local PC, you'll have to edit this accordingly.

In [ ]:
import graphviz
import numpy as np
import os
import pandas as pd
from pathlib import Path
import pydotplus
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
from sklearn import metrics


# We use two different plotting libraries, depending on which kind of plot we want
import matplotlib.pyplot as plt
import seaborn as sns

## Parameters

In [ ]:
# Set an option for Pandas to display smaller floating-point numbers
pd.options.display.float_format = '{:,.2f}'.format

In [ ]:
# Turn on Latex support for matplotlib (if you want it)
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "Bookman"
})

In [ ]:
# Plotting values
background_color = "#fafafa"

In [ ]:
# Machine Epsilon, for computing division
eps = np.finfo(np.float32).eps

# Data Loading and Preprocessing

In [ ]:
from data_loading_functions import load_data, draw_conf_mat

In [ ]:
data = load_data('../data/wisconsin_breast_cancer_data.csv')

# Split apart the pieces of the `data` dictionary
training_values = data['training_values']
training_labels = data['training_labels']
testing_values = data['testing_values']
testing_labels = data['testing_labels']

In [ ]:
feature_names = list(training_values.columns)

# Classifier Creation

For this demonstration, we will create and compare two classifiers: [Naive Bayes](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html#sklearn.naive_bayes.GaussianNB) and [Decision Trees](https://scikit-learn.org/stable/modules/tree.html).

In [ ]:
# Template classifier objects
bayes_clf = GaussianNB()
tree_clf = DecisionTreeClassifier()

# Fit (train) the classifiers
tree_clf.fit(training_values, np.array(training_labels).ravel())
bayes_clf.fit(training_values, np.array(training_labels).ravel())

# Make predictions
tree_predictions = tree_clf.predict(testing_values)
bayes_predictions = bayes_clf.predict(testing_values)

# Accuracy, Precision, and Recall

## Accuracy

Accuracy is the ratio of correctly-labeled samples to the total number of samples:

$$\textrm{accuracy} = \frac{1}{n}\sum_{i=0}^{n}\delta(\hat{y_{i}}, y_{i})$$

where $y_{i}$ is the true label of the $i$-th sample,  $\hat{y_{i}}$ is the classifier prediction, and:

$$\delta(\hat{y_{i}}, y_{i}) = \begin{cases} 1 & \quad \textrm{if } \hat{y_{i}} = y_{i}\\ 0 & \quad \textrm{otherwise}\end{cases} $$

Accuracy ranges from 0 (all samples are incorrectly classified) to 1 (all samples are correctly classified). 

Evaluation based on accuracy typically assumes **balanced classes**: if you have a binary class problem, you can expect that random guessing would result in an accuracy of 50%. 

An accuracy of 0% typically means there's a bug somewhere; it would mean that the classifier is *specifically* outputting the wrong class for each sample, and you could get 100% performance by just doing the opposite of what the classifier says.

## Precision / Recall, Sensitivity / Specificity, And Others

[Data Science in Medicine — Precision & Recall or Specificity & Sensitivity?](https://towardsdatascience.com/should-i-look-at-precision-recall-or-specificity-sensitivity-3946158aace1)

The relationship between these metrics can be confusing, so I've linked an article above to explain it. To summarize:

### Precision and Recall are typically used in *data science*
- *Precision*: Out of all the positively-predicted samples, how many are really positive?
$$\frac{tp}{tp + fp}$$
- *Recall*: Out of all the true positive samples, how many are predicted as positive?
$$\frac{tp}{tp + fn}$$

### Sensitivity and Specificity are typically used in *medical testing*
- *Sensitivity*: Out of all the people who have a disease, how many got positive results?
$$\frac{tp}{tp + fn}$$
- *Specificity*: Out of all the people who do *not* have a disease, how many got negative results?
$$\frac{tn}{tn + fp}$$

### $F$-measure and $F_{1}$ Score

The $F$-measure combines precision and recall into a single value. The most common version of this is the $F_{1}$ score, which assumes equal value of errors (i.e. we want to *balance* precision and recall):

$$F_{1} = \frac{2tp}{2tp + fn + fp}$$

If you want to weight your classifier to favor one metric or the other, you can select a weight $\beta$ which results in:

$$F_{\beta} = \frac{(1 + \beta^{2})tp}{(1+\beta^{2})tp + \beta^{2}fn + fp}$$

If you choose $\beta=2$, it will weight recall higher than precision. If $\beta=0.5$, precision is weighted higher than recall.

### Additional Points

- **Recall** and **Sensitivity** are the same calculation
- If you classify every sample as positive, you will **maximize sensitivity / recall** and **minimize specificity**
- If you classify every sample as negative, you will **maximize specificity** and **minimize sensitivity / recall**
- These metrics are *necessary to use* if you have an imbalanced dataset of positive and negative samples

In [ ]:
print(55 * "=")
print("Decision Trees")
print(55 * "-")
print(metrics.classification_report(testing_labels, tree_predictions, target_names=['Benign', 'Malignant']))

In [ ]:
print(55 * "=")
print("Bayes")
print(55 * "-")
print(metrics.classification_report(testing_labels, bayes_predictions, target_names=['Benign', 'Malignant']))

In [ ]:
draw_conf_mat(testing_labels, tree_predictions)

In [ ]:
draw_conf_mat(testing_labels, bayes_predictions)

# Probabilistic Classification Performance

In [ ]:
bayes_clf = GaussianNB()
bayes_clf.fit(training_values.iloc[:,:5], np.array(training_labels).ravel())
bayes_predictions = bayes_clf.predict(testing_values.iloc[:,:5])

In [ ]:
bayes_predictions

In [ ]:
bayes_probs = bayes_clf.predict_proba(testing_values.iloc[:,:5])

In [ ]:
bayes_probs

In [ ]:
plt.hist(bayes_probs[:,1])
plt.show()

## Review of Probability Classification

Sometimes it's good to evaluate the performance of a probabilistic classification output, like a Bayesian classifier.

Assuming the costs / risks of error are equal, and assuming the priors are the same ($p(\omega_{1}) = p(\omega_{2})$), then $\hat{y} = \omega_{1}$ if $p(\omega_{1}|\mathbf{x}) > p(\omega_{2}|\mathbf{x})$, otherwise $\hat{y}=\omega_{2}$.

Since we're dealing with probabilities:

- $\sum_{i=1}^{2} p(\omega_{i} | \mathbf{x}) = 1 $, and
- $p(\omega_{i}|\mathbf{x}) \geq 0 \qquad\forall i$,

So the above rule is equivalent to saying:

$$\hat{y_{i}} = \begin{cases} \omega_{1} & \quad \textrm{if } p(\omega_{1} | \mathbf{x}) > 0.5\\ \omega_{2} & \quad \textrm{otherwise}\end{cases} $$

(Recall that if $p(\omega_{1} | \mathbf{x}) = 0.5$, then $p(\omega_{2} | \mathbf{x}) = 0.5$ as well, so the classification is ambiguous / randomly selected.) 

## Thresholding

In the above case, our *threshold* of classification is 0.5. Let's denote this as $\tau$ -- now, we can say:

$$\hat{y_{i}} = \begin{cases} \omega_{1} & \quad \textrm{if } p(\omega_{1} | \mathbf{x}) > \tau\\ \omega_{2} & \quad \textrm{otherwise}\end{cases} $$

If $\tau=0.5$, we are assuming that the classifier is "well-tuned", meaning $p(\omega_{i}|\mathbf{x})$ is an accurate description of the likelihood of $\mathbf{x}$ belonging to class $\omega_{i}$.

Alternatively, if $\tau = 0.01$, what happens?

Almost everything is classified as $\omega_{1}$. This is the same as *maximizing sensitivity* and *minimizing specificity*. 

What if $\tau = 0.99$?

Almost everything is classified as $\omega_{2}$. This is the same as *minimizing sensitivity* and *maximizing specificity*.

Since the classification label is dependent on $\tau$, this means that the metrics above also depend on $\tau$. So how can we compare the sensitivity and specificity of probabilistic classifiers?


## Receiver Operating Characteristic (ROC) curves and Area Under the Curve (AUC)

In [ ]:
fpr, tpr, thresholds = metrics.roc_curve(testing_labels, bayes_probs[:,1])
auc = metrics.roc_auc_score(testing_labels, bayes_probs[:,1])

In [ ]:
fpr

In [ ]:
tpr

In [ ]:
thresholds

In [ ]:
# Create the holder for the plots
f, ax = plt.subplots(figsize=(5,5), dpi=300, facecolor=background_color)
ax.set_facecolor(background_color)

ax.plot(fpr, tpr, label=f'AUC: {auc:0.2f}')
ax.scatter(fpr, tpr, label=f'Thresholds', c='r', alpha=0.8)
ax.plot([0, 1], [0, 1], '--k', label='Random')

ax.set(xlabel=r'False Positive Rate (FPR)',
       ylabel=r'True Positive Rate (TPR)',
       title=f'ROC Curve for Bayes Classifier')

ax.legend(frameon=True)
ax.grid(linestyle=':')
plt.tight_layout()

plt.show()